# M3: scalar historical CLV and relationship propagation - Dunnhumby

This seed-42 validation-only screen compares M1 with two scalar-CLV graph interventions and their matching CLV-free relationship controls. All arms use the same binary edge set, train-only signals, propagation strength 0.075, plain BPR, uniform negative sampling, and MIN_ITEM_INTER=1. Test and holdout are not constructed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil, subprocess

REVIEWED_SHA = '44b18d7a934f3421e219ad512b6e7121f9de1aed'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
%cd /content/clv-m2-lightgcn-runner
print('Pinned execution source:', actual_sha)

In [ ]:
import json, torch
from lightgcn_clv_m3_clv_relation import (
    configure_m3_clv_relation_dunnhumby_run,
    preflight_summary,
    run_experiment,
)

cfg = configure_m3_clv_relation_dunnhumby_run()
assert torch.cuda.is_available(), 'Select a GPU runtime before running.'
summary = preflight_summary(cfg)
assert summary['seed'] == 42
assert summary['split'] == 'validation only'
assert summary['target_propagation_strength'] == 0.075
assert summary['eval_test'] is False
assert summary['eval_holdout'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = run_experiment(cfg)

In [ ]:
from IPython.display import display

columns = [
    'model_id', 'role',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'precision@10', 'mean_hits@10', 'hit_value@10',
    'revenue@10', 'revenue@20', 'revenue@50', 'arp@10',
    'coverage@10', 'n_distinct@10', 'exposure_entropy@10',
    'eff_catalog@10', 'top10_share@10', 'top100_share@10',
    'value_alignment',
]
available = [column for column in columns if column in result_df.columns]
validation = result_df[result_df['split'].eq('val')][available]
display(validation.sort_values('model_id'))
print('Screening decision:')
print(json.dumps(result_df.attrs['screening_decision'], ensure_ascii=False, indent=2))
print('Result files:', result_df.attrs['result_paths'])